In [ ]:
# bootstrap: Colab clone + local import of `quantcore` (auto-inserted)
import sys, pathlib
if "google.colab" in sys.modules:
    import os, subprocess
    _slug = "aniryou/full-stack-agentic-engineer"
    _repo = pathlib.Path("/content/full-stack-agentic-engineer")
    if not _repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_repo)], check=True)
    os.chdir(_repo / "04-inference-engine/quantization/quant-core")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
_r = pathlib.Path.cwd().resolve()
while _r != _r.parent and not (_r / "quantcore").exists():
    _r = _r.parent
if str(_r) not in sys.path:
    sys.path.insert(0, str(_r))
del _r

# 05 · Choosing a scheme

**Tier:** T0 — arithmetic on a model config and a GPU spec, a few seconds. Every latency printed here is
**SIMULATED** (a roofline model: max(bytes / bandwidth, FLOPs / peak) with assumed efficiencies); GPU figures
are dense datasheet values as of September 2026 `(verify)`. Measuring the same schemes on a real GPU is the
lab's notebook 02 (T1).

## The one-minute version
Choose in three steps. **What bounds the workload?** Decode streams every weight each step, so it is bound by
weight bytes (and, at long context and high batch, KV bytes); prefill is bound by FLOPs; concurrency is bound by
KV memory. **What can the GPU execute natively?** Weight-only formats (W4A16, W8A16) run anywhere from Turing up
and cut bytes, not FLOPs; FP8 W8A8 needs Ada or newer; FP4 W4A4 needs Blackwell; INT8 W8A8 is gone on
Blackwell; an FP8 checkpoint on an A100 runs as weight-only FP8 — `cost.supported` encodes vLLM's rules. **What
accuracy can you afford?** Rank schemes from least to most aggressive and take the first that meets the
latency and concurrency targets *and* passes the eval (notebook 03, primer §8). After this notebook you can
produce and defend that table for any GPU and model in the repo, and put a cost per million tokens on it.

Primer: `../PRIMER.md` §1 *Why quantize* and §10 *Choosing a scheme*; the roofline is layer 01's
(`01-hardware-gpu-fabric/roofline-and-fabric/PRIMER.md` §2–3), the cost formula its §8.

> **Exercise cells** contain `# YOUR CODE HERE` — replace it, then run the **Check** cell below it. A check prints ✅ when it passes. The finished version is in `solutions/`.

In [ ]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")   # small matrices: one BLAS thread is fastest, and safe on busy machines

from quantcore import cost as C, eval as E, quantize_model, TinyModel

L4, H100, T4 = C.GPUS["L4"], C.GPUS["H100-SXM"], C.GPUS["T4"]
m8 = C.MODELS["llama-3.1-8b"]


def show(rows, session=2000):
    print(f"{'scheme':11} {'KV':>3} {'runs as':11} {'weights':>8} {'dec b=1':>8} {'dec b=32':>9} {'prefill':>8} {'sess':>5}  note")
    for r in rows:
        if "prefill_ms" in r:
            print(f"{r['scheme']:11} {r['kv_bits']:3d} {r['runs_as']:11} {r['weights_gb']:6.2f}GB {r['decode_1_ms']:6.1f}ms "
                  f"{r['decode_b_ms']:7.1f}ms {r['prefill_ms']:6.0f}ms {r['sessions']:5d}  {r['note']}")
        else:
            print(f"{r['scheme']:11} {r['kv_bits']:3d} {'-':11} {r['weights_gb']:6.2f}GB {'':38}  {r['note']}")
    print("(SIMULATED: 80% of bandwidth, 60% of peak FLOP/s, 2 ms per step; decode at 1,000 tokens of context; "
          f"prefill of 1,800 tokens; sessions of {session:,} tokens)")

## Worked example 1 — the serving-engine §8 table, reproduced
serving-engine PRIMER §8 prints what each scheme buys for Llama-3.1-8B on a 24 GB L4 from `minengine.perf`.
`quantcore.cost.step_cost` is the same model; with the L4 defined as minengine defines it (FP8 peak exactly 2×
bf16) the numbers match to the digit (`tests/test_repo_numbers.py`). The table below uses `roofline.specs`'
242.5 TFLOP/s FP8 peak instead of 242; at this precision it prints the same.

In [ ]:
show(C.table(L4, m8, schemes=["bf16", "w8a16-fp8", "w4a16", "w8a8-fp8"]))

Read it by bound. Decode at batch 1 is the weight read: 16.1 → 9.1 → 5.7 GB is 65 → 36 → 22 ms, not 3.9×,
because the 16-bit LM head, the KV read and the step overhead do not shrink. Prefill is compute: only FP8 W8A8
halves it. And sessions follow the bytes left for KV: FP8 KV doubles them at any weight format.

## Worked example 2 — one GEMM on the roofline: where W4A16 stops paying
vllm-internals §8.1 prices Llama-3.1-8B's `down_proj` (K = 14,336, N = 4,096) on an L4 for M tokens in the
step. `cost.gemm_time` reproduces it (W4A16 at 4.16 bits per weight, as that table counts).

In [ ]:
for M in (1, 16, 64, 128, 256, 400, 462, 2048):
    row = [C.gemm_time(M, 14336, 4096, L4, s, w_bits=4.16 if s == "w4a16" else None) for s in ("bf16", "w4a16", "w8a8-fp8")]
    print(f"M = {M:5d}: " + "  ".join(f"{s} {r['t'] * 1e6:7.0f} us ({r['bound'][:3]})" for s, r in zip(("BF16", "W4A16", "FP8"), row))
          + f"   W4A16 speedup {row[0]['t'] / row[1]['t']:.2f}x")
for g in (L4, H100):
    print(f"{g.name}: W4A16 turns compute-bound above {C.crossover_tokens(14336, 4096, g, w_bits=4.16):.0f} tokens per step, "
          f"BF16 above {C.crossover_tokens(14336, 4096, g, 'bf16'):.0f}")

A weight-only kernel does BF16 math on dequantized weights. Up to ~120 tokens per step on an L4 (~85 on an H100)
it is byte-bound and keeps the full byte ratio, ~3.8×. There it hits the BF16 compute ceiling while BF16 itself is
still byte-bound, so from there its speedup shrinks — 1.7× at 256 tokens — and is gone where BF16 turns
compute-bound too, ~460 tokens on an L4 (~330 on an H100). So decode batches of a few hundred still gain from
INT4; long prefill chunks (512 and up) do not, and real kernels lose sooner because dequantization is not free
(NVIDIA's ModelOpt measured weight-only NVFP4 slower than BF16 in 10 of 12 shapes on Blackwell, its QAD note of
2026-09-16, verify). FP8 W8A8 halves both ceilings and helps at every M.

## Worked example 3 — what a checkpoint runs as, per GPU generation

In [ ]:
print(f"{'':13}" + "".join(f"{s:>13}" for s in C.SCHEMES) + f"{'FP8 KV':>9}")
for g in C.GPUS.values():
    cells = [C.supported(g, s)["runs_as"] or "no" for s in C.SCHEMES]
    print(f"{g.name:13}" + "".join(f"{c:>13}" for c in cells) + f"{'yes' if C.supported(g, 'bf16', 8)['kv'] else 'no':>9}")

Read down the FP8 W8A8 column: an FP8 checkpoint loads everywhere from Turing up, but below Ada it is a
weight-only model (Marlin FP8) with BF16 math — a memory win only. NVFP4 is W4A4 only on Blackwell (SM100/SM120,
CUDA ≥ 12.8); elsewhere it runs as weight-only 4-bit. INT8 W8A8 is the Turing/Ampere prefill lever and is not
supported from compute capability 10.0. A T4 has no FP8 KV cache in any vLLM backend. (vLLM 0.30.0 and main,
from the kernels' capability checks; verify on your version — the log line `Selected <kernel> for <module>` is the
truth.)

## Worked example 4 — three deployments

In [ ]:
print("== a free Colab/Kaggle T4, Qwen2.5-1.5B (16 GB, fp16 only, no FP8 KV)")
show(C.table(T4, C.MODELS["qwen2.5-1.5b"], kv=(16,), schemes=["bf16", "w8a8-int8", "w4a16"]))
print("\n== an H100 80GB, Llama-3.1-70B")
show(C.table(H100, C.MODELS["llama-3.1-70b"], schemes=["bf16", "w8a8-fp8", "w4a16"], session=4000), session=4000)
print("\n== a B200, Llama-3.1-8B (verify: Blackwell FP4 figures)")
show(C.table(C.GPUS["B200"], m8, kv=(8,), schemes=["bf16", "w8a8-fp8", "w8a8-int8", "w4a4-nvfp4"]))

On a T4 a 1.5B model is small enough that quantization is about speed: INT4 for decode, INT8 W8A8 for prefill.
On an H100 a 70B model in 16-bit does not fit at all. In FP8 it fits, but with this model's round memory inputs
(0.9 × 80 GB − 1 GB) there is no room left for a 4K session; at vLLM's defaults on the 79.65 GiB an H100 reports
(the lab's `quantlab.kv.size`) there is room for 2 such sessions, or 4 with FP8 KV — either way no useful
concurrency. On one GPU only a 4-bit format serves it; FP8 means two GPUs with tensor parallelism, or an H200's
141 GB. On a B200, NVFP4 W4A4 is the 4-bit format that also cuts prefill FLOPs — if the model survives 4-bit
activations (notebook 04, worked example 4).

## Worked example 5 — what it costs per million tokens
`$/M tokens = $/GPU-hour ÷ (tokens/s × 3600 × utilisation) × 10⁶` (layer 01 §8.1). Prices are the research
snapshot's GCP list prices, us-central1, September 2026 — L4 ~$0.70/hr, H100 ~$11/GPU-hr on demand — `(verify)`;
`COMPUTE.md` keeps them current.

In [ ]:
for g, price in ((L4, 0.70), (H100, 11.0)):
    for s, kv in (("bf16", 16), ("w8a8-fp8", 8), ("w4a16", 8)):
        batch = min(256, C.sessions(g, m8, s, 2000, kv))
        tps = batch / C.step_cost(g, m8, s, [(1000, 1)] * batch, kv)["t"]
        print(f"{g.name:9} {s:9} KV{kv:<2} batch {batch:3d}: {tps:7,.0f} tok/s  ${C.cost_per_million(price, tps):.3f}/M at 100% "
              f"(${C.cost_per_million(price, tps, 0.6):.3f} at 60%)  SIMULATED")

Quantization lowers cost twice: fewer bytes per step, and more sessions, so a bigger batch shares each step.
On the L4 the batch is capped by KV memory (17 sessions in bf16), which is why FP8 weights plus FP8 KV cut the
cost per token ~6× there, not 2×: a 5× larger batch in a step that is no longer. On the H100, BF16 fits 209
sessions and FP8 reaches the 256 cap used here; FP8 roughly halves the cost.

## Worked example 6 — the accuracy gate
The eval decides which rows are allowed at all. On the tiny model (notebook 03) with a budget of KL ≤ 0.05 nats
and an accuracy drop within two standard errors of the difference (`E.diff_stderr`, unpaired; the paired
McNemar z on the flips, `E.paired_z`, is the sharper test — primer §8):

In [ ]:
tm = TinyModel()
Xt, yt = tm.sample(4000, "test")
Xc, _ = tm.sample(256, "calib")
ref = tm.forward(Xt)
for label, q in (("INT8 RTN", quantize_model(tm, "rtn", 8, None)), ("INT4 RTN g32", quantize_model(tm, "rtn", 4, 32)),
                 ("INT4 GPTQ g32", quantize_model(tm, "gptq", 4, 32, calib=Xc))):
    r = E.compare(ref, q.forward(Xt), yt)
    print(f"{label:14} KL {r['kl']:.4f}  acc {r['acc']:.1%} (fp {r['acc_ref']:.1%}; drop {r['acc_ref'] - r['acc']:+.1%} vs "
          f"2 x {r['diff_stderr']:.1%})  paired z {r['paired_z']:+.1f}  within budget: {E.within_budget(r, max_kl=0.05)}")

INT4 is allowed only with GPTQ here — the recipe is part of the scheme. GPTQ's 0.6-point drop is inside the
unpaired noise bar, while the paired flips (85 lost, 61 gained, z = −2.0) say it is a small but probably real loss:
a budget should say which test it means. `choose(rows, allowed=...)` takes that set.

## Exercise 5.1 — the W4A16 crossover by hand
For a linear with K inputs and N outputs, M tokens: FLOPs `2MKN`, bytes `K·N·w + M·(K·a + 2N)` with `w` bytes
per weight (4.16 bits → 0.52 B) and `a = 2` bytes per activation; the time is the larger of FLOPs / peak and
bytes / bandwidth. Solve for the M where the two are equal, for the L4 (121 TFLOP/s bf16, 0.30 TB/s) and the H100
(989.4, 3.35). Set `m_l4` and `m_h100`.

In [ ]:
K_, N_ = 14336, 4096
# YOUR CODE HERE
raise NotImplementedError("your turn")

In [ ]:
assert round(m_l4) == 120 and round(m_h100) == 85
print(f"✅ {m_l4:.0f} tokens on an L4, {m_h100:.0f} on an H100: past that W4A16's edge shrinks, and where BF16 turns "
      "compute-bound too (~460 and ~330) it is a memory format, not a speed format")

## Exercise 5.2 — what does an FP8 W8A8 checkpoint run as?
Without calling `cost.supported`, fill `runs_as` for an FP8 W8A8 checkpoint (`"w8a8-fp8"`) on each GPU with the
scheme it actually executes: `"w8a8-fp8"` or `"w8a16-fp8"` (weight-only), from the compute capabilities
(T4 7.5, A100 8.0, L4 8.9, H100 9.0, B200 10.0).

In [ ]:
runs_as = {"T4": None, "A100-80GB": None, "L4": None, "H100-SXM": None, "B200": None}
# YOUR CODE HERE
raise NotImplementedError("your turn")

In [ ]:
assert runs_as == {g: C.supported(C.GPUS[g], "w8a8-fp8")["runs_as"] for g in runs_as}
print("✅ FP8 tensor cores start at Ada (sm_89): on a T4 or A100 the same checkpoint saves memory but not FLOPs")

## Exercise 5.3 — pick a scheme for a free T4
Serve Llama-3.1-8B on a T4 (16-bit KV only) with at least **20** concurrent 2,000-token sessions and a 1,800-token
prefill in at most **700 ms**. From `C.table(T4, m8, kv=(16,))`, pick the least aggressive scheme (order
`C.ACCURACY_ORDER`) that meets both, and set `choice` to its name.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError("your turn")

In [ ]:
assert choice == "w4a16"
print("✅ w4a16: 16-bit weights do not fit a T4, 8-bit weights leave room for only 16 sessions; INT4 leaves 29. "
      "On a T4 the INT4 checkpoint is the only way to serve an 8B model at all, and GPTQ/AWQ is how it stays accurate")

## Exercise 5.4 — cost per million tokens
The L4 serving Llama-3.1-8B with FP8 weights and FP8 KV runs batch 32 in 44.2 ms per decode step (SIMULATED).
At $0.70 per hour and 60% utilisation, what does a million output tokens cost? Set `usd_per_m`.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError("your turn")

In [ ]:
assert abs(usd_per_m - C.cost_per_million(0.70, 32 / 0.0442, 0.6)) < 1e-9 and 0.44 < usd_per_m < 0.46
print(f"✅ ${usd_per_m:.3f} per million output tokens (SIMULATED throughput, list price (verify))")

## Exercise 5.5 — a 70B model on one 80 GB GPU
For Llama-3.1-70B on one H100, fill `fits`: scheme → the number of 4,000-token sessions with FP8 KV (0 means the
weights leave no room), for `"bf16"`, `"w8a8-fp8"`, `"w4a16"` and `"w4a4-nvfp4"`. Then set `smallest_ok` to the
least aggressive scheme that serves at least 32 such sessions.

In [ ]:
fits, smallest_ok = {}, None
# YOUR CODE HERE
raise NotImplementedError("your turn")

In [ ]:
assert fits == {"bf16": 0, "w8a8-fp8": 0, "w4a16": 48, "w4a4-nvfp4": 43} and smallest_ok == "w4a16"
print("✅ 141 GB in bf16, 72.7 GB in FP8 (no room left at these round inputs; 2-4 sessions at vLLM's real budget), "
      "39.5 GB in INT4: on one H100 a 70B model is a 4-bit model; FP8 means two GPUs with tensor parallelism, or an H200")

## In a design review
**The two-minute version.** "We chose the scheme from the bottleneck, the hardware and the eval, in that order.
Our traffic is decode-heavy chat on L4s, so weight bytes and KV memory decide: FP8 W8A8 halves the weights and
halves prefill on Ada's FP8 tensor cores, and an FP8 KV cache doubles the sessions — 87 instead of 17 for an 8B
model on a 24 GB L4, simulated — which is also what cuts cost per token ~6×: a bigger batch shares every weight
read. INT4 weight-only would decode faster still but does not speed prefill (above ~120 tokens per step on an L4
its GEMM is BF16-math-bound, and by ~460 plain BF16 has caught up), so we keep it for memory-bound cases: a T4, or a 70B on one 80 GB GPU. We checked
what each checkpoint runs as on each generation — FP8 on an A100 is weight-only, NVFP4 is W4A4 only on Blackwell,
INT8 W8A8 disappears on Blackwell — and every scheme passed the eval budget before it reached this table."

**Drills**
1. *Why does INT4 give 3× faster decode but no faster prefill on an L4?* — Decode is a weight read (bytes ÷ 4);
   prefill is BF16 math on dequantized weights: compute-bound above ~120 tokens per step, and no faster than
   BF16 past ~460, where BF16 is compute-bound too.
2. *We have A100s. Is FP8 worth it?* — For memory and decode bytes, yes (weight-only FP8 via Marlin); for prefill
   no — A100s have no FP8 tensor cores; INT8 W8A8 (with SmoothQuant) is the A100 prefill lever.
3. *Why did FP8 cut the L4's cost per token 6×, not 2×?* — Fewer bytes per step, and five times the sessions
   (FP8 weights free memory, FP8 KV halves each session), so each step serves a much larger batch.